In [ ]:
%pip install -U transformers torch torchvision accelerate

## Local Inference on GPU 
Model page: https://huggingface.co/google/gemma-4-E2B-it

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/gemma-4-E2B-it)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Load model directly
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

model_path = "google/gemma-4-E2B-it"
processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForMultimodalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto",
).eval()

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/apps/sample-data/GoldenGate.png"},
            {"type": "text", "text": "What is shown in this image?"},
        ],
    },
]
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=64)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
print(response)
print(processor.parse_response(response))
model_parameter_devices = sorted({str(p.device) for p in model.parameters()})
print("model parameter devices =", model_parameter_devices)
